# MMIS 671 Final Exam: Supervised Machine Learning using *ScikitLearn* Classifiers

## Import libraries

In [ ]:
import pandas as pd # for data handling
import numpy as np # for computations
import time # to record training and testing time
import matplotlib.pyplot as plt # for plotting

# import metrics to evaluate models
from sklearn.metrics import accuracy_score # accuracy
from sklearn.metrics import confusion_matrix # confusion matrix
from sklearn.metrics import classification_report # precision, recall, f1-score

# scikit-learn classifiers
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression # Logistic Regression
from sklearn.neural_network import MLPClassifier # Neural Network

from sklearn.model_selection import GridSearchCV

from google.colab import drive 
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Task 0.
Read data from the CSV files “*MMIS671.final.q1.train.csv*”, “*MMIS671.final.q1.test.csv*”, and “*MMIS671.final.q1.new.csv*” into pandas dataframes *train*, *test*, and *new*, respectively. Confirm that the dataframes contain the correct number of rows and columns.

Report the class distribution of ‘y’ in train and test by specifying the proportion of examples in each class. Round the proportions to 4 decimal places. 


### Read data into dataframes

In [ ]:
# Read data into pandas dataframes
train = pd.read_csv('/content/drive/My Drive/Colab Notebooks/MMIS671.final.q1.train.csv')
test = pd.read_csv('/content/drive/My Drive/Colab Notebooks/MMIS671.final.q1.test.csv')
new = pd.read_csv('/content/drive/My Drive/Colab Notebooks/MMIS671.final.q1.new.csv')

features = list(train)[1:]
print("Training: %d rows, %d columns" %train.shape)
print("Test: %d rows, %d columns" %test.shape)
print("Unlabeled: %d rows, %d columns" %new.shape)
print("Number of features = %d" %len(features))

Training: 60000 rows, 61 columns
Test: 12000 rows, 61 columns
Unlabeled: 40 rows, 61 columns
Number of features = 60


### Class distribution of *y*

In [ ]:
dist_y = pd.DataFrame() # dataframe with class distribution of y
dist_y['y'] = [0, 1, 2, 3] # values of y
dist_y['Proportion in train'] = train.y.value_counts(normalize=True, sort=False)
dist_y['Proportion in test'] = test.y.value_counts(normalize=True, sort=False)
dist_y.to_csv('dist_y.csv', index=False) # save results
dist_y.round(4) # display results

,y,Proportion in train,Proportion in test
0,0,0.2496,0.2496
1,1,0.2501,0.2502
2,2,0.2504,0.2504
3,3,0.2499,0.2498


## Task 1.
Train ScikitLearn classifiers (with default parameters) using the 60,000 labeled examples in train. Report the time it takes to train the model. 

Predict ‘*y*’ for the  12,000 examples in test using the trained classifier. Report the time it takes to predict the 12,000 test examples.

Report the classification accuracy of your trained model for the 12,000 test examples.  

Classifiers:

- GaussianNB
- DecisionTreeClassifier
- RandomForestClassifier
- KNeighborsClassifier
- SVC
- LogisticRegression
- MLPClassifier





### Specify classifiers
We shall create a dictionary called *classifiers* to contain  classifiers with default parameters. 

In [ ]:
classifiers = {'DT': DecisionTreeClassifier(),
               'RF': RandomForestClassifier(),} # dictionary with classifiers 

### Train and evaluate classifiers 

In [ ]:
results = [] # list with results

for c in classifiers: # for each classifier
    print("Classifier", c) # classifier being used
    clf = classifiers[c] # create classifier
    print(clf) # show hyper-parameters for classifier
    
    # train classifier 
    st = time.time() # start time for training
    clf.fit(train[features], train.y) # train model
    train_time = time.time() - st # training time
    print("Training time = %4.3f seconds" %train_time)
    
    # predict test cases using trained classifier
    st = time.time() # start time for prediction
    pred = clf.predict(test[features]) # predict y using trained model
    pred_time = time.time() - st # training time
    print("Prediction time = %4.3f seconds" %pred_time)

    # Compute prediction accuracy on test examples
    acc = accuracy_score(test.y, pred) # compute accuracy
    print("Accuracy on test examples = %4.4f" %acc)

    # record results for classifier
    results.append([c, acc, train_time, pred_time])

    print(80*"=" + '\n')

# Results summary
cols = ['Classifier', 'Accuracy', 'Training_time', 'Prediction_time'] # column headers for results
task1_df = pd.DataFrame(results, columns=cols) # dataframe with results
task1_df.to_csv('task1_results.csv', index=False) # save results
task1_df # display results

Classifier DT
DecisionTreeClassifier(ccp_alpha=0.0, class_weight=None, criterion='gini',
                       max_depth=None, max_features=None, max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, presort='deprecated',
                       random_state=None, splitter='best')
Training time = 14.301 seconds
Prediction time = 0.024 seconds
Accuracy on test examples = 0.9123

Classifier RF
RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       

,Classifier,Accuracy,Training_time,Prediction_time
0,DT,0.912333,14.301259,0.023716
1,RF,0.964667,65.397157,0.386982


## Task 2. 
Based on the results on task 1, identify those classifiers that result in a classification accuracy of greater than 0.92 for the test examples. For each such classifier, find a good set of hyper-parameters through 5-fold cross-validation (using only data from train). 

Choose one of these classifiers (with desired hyper-parameters) based on accuracy, training time, prediction time, and interpretability considerations and present the model as your *chosenModel*. Provide a brief rationale for your choice.

Train *chosenModel* using the 60,000 labeled examples from train and use this trained model to predict the output classes for the examples in test. 

Report the classification *accuracy* of *chosenModel* for the 12,000 test examples.

Report the *confusion matrix* for the 12,000 test examples:
	

Report *precision*, *recall*, and *F1-score* (rounded to 4 decimal places) for the 12,000 test examples (using sklearn’s *classification_report*).

### Find good parameter for chosen model

I am going to find a good set of good hyper-parameter values through 5-fold cross-validation using grid search for the DecisionTreeClassifier. You may do the same for other classifiers too. 


In [58]:
parameters = [{'max_leaf_nodes': range(2,10),
               'criterion': ['gini', 'entropy']}] # trees with 2 to 9 leaf nodes 
# define model for grid search
gs = GridSearchCV(DecisionTreeClassifier(),
                  parameters,
                  n_jobs=-1, verbose=1)  # model for grid search

gs.fit(train[features], train.y) # evaluate hyper-parameters

print("\nBest parameters found:")
print(gs.best_params_) # best hyperparameter values

print("\nGrid scores:")
means = gs.cv_results_['mean_test_score'] # mean accuracy with folds
stds = gs.cv_results_['std_test_score'] # standard deviation of accuracies
# for each hyperparameter combination show mean +/- 2 standard-deviations 
for mean, std, params in zip(means, stds, gs.cv_results_['params']):
    print("%0.4f (+/-%0.04f) for %r" %(mean, std * 2, params)) 

Fitting 5 folds for each of 16 candidates, totalling 80 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   29.5s
[Parallel(n_jobs=-1)]: Done  80 out of  80 | elapsed:   55.0s finished



Best parameters found:
{'criterion': 'gini', 'max_leaf_nodes': 9}

Grid scores:
0.3800 (+/-0.0029) for {'criterion': 'gini', 'max_leaf_nodes': 2}
0.5475 (+/-0.0040) for {'criterion': 'gini', 'max_leaf_nodes': 3}
0.6342 (+/-0.0049) for {'criterion': 'gini', 'max_leaf_nodes': 4}
0.7307 (+/-0.0052) for {'criterion': 'gini', 'max_leaf_nodes': 5}
0.7769 (+/-0.0051) for {'criterion': 'gini', 'max_leaf_nodes': 6}
0.8235 (+/-0.0031) for {'criterion': 'gini', 'max_leaf_nodes': 7}
0.8584 (+/-0.0044) for {'criterion': 'gini', 'max_leaf_nodes': 8}
0.8797 (+/-0.0036) for {'criterion': 'gini', 'max_leaf_nodes': 9}
0.4395 (+/-0.0024) for {'criterion': 'entropy', 'max_leaf_nodes': 2}
0.5258 (+/-0.0058) for {'criterion': 'entropy', 'max_leaf_nodes': 3}
0.5713 (+/-0.0042) for {'criterion': 'entropy', 'max_leaf_nodes': 4}
0.6364 (+/-0.0054) for {'criterion': 'entropy', 'max_leaf_nodes': 5}
0.7165 (+/-0.0052) for {'criterion': 'entropy', 'max_leaf_nodes': 6}
0.7613 (+/-0.0112) for {'criterion': 'entropy'

**Important Features**

In [ ]:
imp_features = []
for imp, f in sorted(zip(chosenModel.feature_importances_, features), 
                     reverse=True): # (feature, importance) in descending order
    if imp > 0.005: # mininimum threshhold importance
        print(f, imp)
        imp_features.append(f)
imp_features

x25 0.6337747219976502
x45 0.36622527800234983


['x25', 'x45']

In [ ]:
rf_if = RandomForestClassifier()
rf_if.fit(train[imp_features], train.y)

RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

In [ ]:
%%time
train[imp_features].head(10)

CPU times: user 2.17 ms, sys: 11 µs, total: 2.18 ms
Wall time: 2.13 ms


,x25,x45
0,0.678,0.860
1,0.846,0.497
2,0.226,0.300
3,0.259,0.646
4,0.352,0.605
5,0.418,0.418
6,0.171,0.762
7,0.292,0.609
8,0.626,0.259
9,0.734,0.387


### Train *chosenModel*

### Evaluate *chosenModel*

In [ ]:
pred = chosenModel.predict(train[features]) # predict classes for test examples
print("Accuracy = %4.4f" %accuracy_score(train.y, pred))
print("Classification Report:")
print(classification_report(train.y, pred, digits=4))
print("Confusion matrix")
cm = pd.DataFrame(confusion_matrix(train.y, pred)) # confusion matrix as dataframe
cm.to_csv("DT_conf_matrix.csv")
cm


Accuracy = 0.6348
Classification Report:
              precision    recall  f1-score   support

           0     0.7561    0.3744    0.5008     14974
           1     0.9205    0.7107    0.8021     15007
           2     0.4437    0.9283    0.6004     15026
           3     0.8228    0.5246    0.6407     14993

    accuracy                         0.6348     60000
   macro avg     0.7358    0.6345    0.6360     60000
weighted avg     0.7356    0.6348    0.6361     60000

Confusion matrix


,0,1,2,3
0,5606,271,8965,132
1,401,10665,2487,1454
2,489,480,13949,108
3,918,170,6040,7865


## Task 3.
Use the trained *chosenModel* to predict the output classes for the 100 unlabeled examples in *new*.


In [ ]:
new_pred = pd.DataFrame() # dataframe with predicted classes for unlabeled examples
new_pred['ID'] = new.ID # identifiers for unlabeled examples
new_pred['y'] = chosenModel.predict(new[features]) # predicted labels
new_pred.to_csv('task3_results.csv', index=False) # save results
new_pred # show results

,ID,y
0,1,2
1,2,0
2,3,0
3,4,0
4,5,2
5,6,0
6,7,0
7,8,2
8,9,0
9,10,2
